# TN2 — Tầm nhìn của DS-TCN 192 kênh

## Câu hỏi

Cấu hình nền có tầm nhìn **61 mẫu** trên cửa sổ vào **200 mẫu** — chỉ thấy 30
phần trăm cửa sổ, tức 1,2 giây ở 50 Hz, chưa tới một nhịp thở dài khoảng 4 giây.

Điểm hiện tại của nó: **0,762714 (1 seed)**.

**Mở rộng tầm nhìn tới quanh 200 mẫu có làm khá lên không?**

## Thang thí nghiệm

Giữ nguyên **4 khối**, chỉ đổi bề rộng kernel. Ở cùng số khối, kernel tăng làm
trọng số depthwise tăng, nhưng phần pointwise mới chiếm đa số — nên số tham số
thay đổi ít:

| kernel | tầm nhìn | phủ cửa sổ | tham số | so nền |
|---:|---:|---:|---:|---:|
| 3 | 61 | 30% | 307.801 | — *(đã chạy)* |
| **5** | 121 | 60% | **310.873** | +1,0% |
| **7** | 181 | 90% | **313.945** | +2,0% |
| **9** | 241 | 100% | **317.017** | +3,0% |

**Tầm nhìn gấp 3,9 lần mà tham số chỉ chênh 3,0 phần trăm.** Nếu điểm đổi
theo thang này thì gần như chắc do tầm nhìn, không do sức chứa.

Cách khác — tăng số khối — làm tham số nhảy tới 49 phần trăm, đổi hai biến cùng
lúc, không tách được.

Năm cấu hình bắc qua mốc 200: hai dưới (121, 181), một vừa đủ (241), hai thừa
Lên tới 241 rồi phẳng nghĩa là phủ hết cửa sổ là đủ; còn lên tiếp
nghĩa là tầm nhìn dài hơn cửa sổ vẫn có ích.

`k=3` không chạy lại vì đã có.

## Chạy vòng sàng lọc

Mỗi cấu hình **một fold `val_KL`** — train ABCDEF, chấm K và L, 218.088 cửa sổ.
Năm cấu hình mất khoảng **2,5 giờ** thay vì gấp bốn.

**Hai điều bắt buộc nhớ**, đo trên tám cấu hình TN1 có đủ ba seed:

**1. Điểm một fold không so được với `cv_score` bốn fold.** `val_KL` là fold dễ
nhất; điểm trên nó cao hơn trung bình **+0,040**, mức chênh không đều, từ +0,020
tới +0,070 tuỳ cấu hình.

**2. Thứ hạng có thể đảo.** BiLSTM-41 đứng chót trên bốn fold nhưng hạng ba nếu
chỉ nhìn `val_KL`; DS-TCN-64 thì từ hạng sáu xuống chót.

Vòng này dùng để **loại**, không dùng để **chọn**. Chi tiết: `docs/SANG_LOC.md`.

`run_cv.py` **không ghi dòng `TONG`** khi thiếu fold, nên số sàng lọc không lọt
vào bảng `compare_cv`. Muốn xác nhận thì chạy lại **bỏ `--folds`** — fold đã
xong được bỏ qua, dòng `TONG` tự có.

## Mốc để đặt cạnh

So **một fold với một fold**. Điểm `val_KL` seed 0:

| | tham số | val_KL |
|---|---:|---:|
| LSTM-352 | 1.502.713 | 0,8257 |
| LSTM-67 | 56.908 | 0,8211 |
| DS-TCN-64 (TN1 gốc) | 56.281 | 0,7723 |

Nhóm kết quả `tn2_rf`, tách khỏi `tn2` của thí nghiệm RevIN.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn. Cờ `--folds` và `--norm none` chỉ có ở bản mới.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q --branch submission --single-branch https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục kết quả đã có từ Drive.

**Ô clone ở trên xoá `runs/`.** Không có bước này thì fold đã chạy bị train lại.
Chưa có kết quả cũ thì ô này không in gì.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn2_rf_*c192*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm năm bản cài đặt

Mỗi lệnh in tầm nhìn ở mục 7, đối chiếu với bảng đầu notebook.

**Đọc dòng cuối mỗi lệnh trước khi chạy tiếp.** Phải là `TẤT CẢ ĐẠT` —
notebook không tự dừng khi lệnh trượt.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 11 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 13 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Chạy năm cấu hình, một fold, một seed

Tên cấu hình: `ds_tcn_c192_k<K>_n4_none_do0.2_dpel_mse_corr0.9_seed0`.

Mỗi lệnh khoảng **30 phút**. Kernel lớn hơn thì nặng hơn một chút.

**kernel 5 — tầm nhìn 121, 310.873 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

**kernel 7 — tầm nhìn 181, 313.945 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

**kernel 9 — tầm nhìn 241, 317.017 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn2_rf --out tn2_rf_ds_tcn_c192

## 5. Đọc kết quả

`compare_cv` sẽ **không** hiện gì vì thiếu fold nên không có dòng `TONG`. Đó là
chủ ý. Ô dưới đọc thẳng `summary.csv`.

In [ ]:
import csv, re
RF = {3: 61, 5: 121, 7: 181, 9: 241, 11: 301, 13: 361}
rows = [r for r in csv.DictReader(open("runs/tn2_rf/summary.csv"))
        if r["fold"] == "val_KL" and "_c192_" in r["run_id"]]
for r in sorted(rows, key=lambda r: int(re.search(r"_k(\d+)_", r["run_id"]).group(1))):
    k = int(re.search(r"_k(\d+)_", r["run_id"]).group(1))
    print("  kernel", k, " tầm nhìn", RF[k], " ", r["n_params"], "tham số  ", r["score_macro"])

## 6. Đường cong tầm nhìn

Vạch đỏ là bề rộng cửa sổ vào.

In [ ]:
import matplotlib.pyplot as plt
d = {RF[int(re.search(r"_k(\d+)_", r["run_id"]).group(1))]: float(r["score_macro"])
     for r in rows}
x = sorted(d); plt.plot(x, [d[i] for i in x], "o-")
plt.axvline(200, ls="--", c="r", label="cửa sổ vào 200 mẫu")
plt.xlabel("tầm nhìn (mẫu)"); plt.ylabel("điểm val_KL"); plt.legend(); plt.grid(alpha=.3)

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()